In [ ]:
import pandas as pd 
import numpy as np 

In [ ]:
sequence_client_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile//2026//Valuation data\clients_final_df.csv"
sequence_client_df = pd.read_csv(sequence_client_path, sep=';')

sequence_client_df = sequence_client_df[['File', 'Listed', 'Principal']]

# pārveidojam formātu no string uz datetime, lai varētu veikt aprēķinus
sequence_client_df["Listed"] = pd.to_datetime(
    sequence_client_df["Listed"],
    format="%Y-%m-%d",
    errors="coerce"
)

# pievienojam kolonnu "payment_threshold", kas ir 2.78% no Principal vērtības
# plānotā veiksmīga parāda atguve 36 mēnešu laikā ir 2.78% no Principal vērtības, tāpēc tiek izmantota šī vērtība kā maksājumu slieksnis 100%/36
# payment_threshold = principal_prct_for_payment_threshold
sequence_client_df['principal_prct_for_payment_threshold'] = sequence_client_df['Principal'] * 0.0278

# 3 gadi no iegādes datuma (maksimālais periods), lai noteiktu periodu, kurā tiek vērtēta maksājumu aktivitāte
sequence_client_df["observation_end"] = sequence_client_df["Listed"] + pd.DateOffset(years=3)

In [ ]:
sequence_transaction_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile//2026//Valuation data\transactions_cl_df.csv"
sequence_transaction_df = pd.read_csv(sequence_transaction_path, sep=';')

In [ ]:
sequence_transaction_df = sequence_transaction_df[
    ["File", "Payment date", "Payment_amount"]
].copy()

sequence_transaction_df["Payment date"] = pd.to_datetime(
    sequence_transaction_df["Payment date"],
    format="%Y-%m-%d",
    errors="coerce"
)

clients_with_payments = sequence_client_df.merge(
    sequence_transaction_df,
    on="File",
    how="left"
)


In [ ]:
# Atstājam tikai maksājumus pēc Listed
clients_with_payments_after_listed = clients_with_payments[
    clients_with_payments["Payment date"].isna()
    |
    (clients_with_payments["Payment date"] >= clients_with_payments["Listed"])
].copy()

In [ ]:
clients_with_payments_after_listed["is_significant_payment"] = (
    clients_with_payments_after_listed["Payment date"].notna()
    &
    (
        clients_with_payments_after_listed["Payment_amount"]
        >= clients_with_payments_after_listed["principal_prct_for_payment_threshold"]
    )
).astype(int)

In [ ]:
first_significant_payment_df = (
    clients_with_payments_after_listed[
        clients_with_payments_after_listed["is_significant_payment"] == 1
    ]
    .sort_values(["File", "Payment date"], na_position="last")
    .groupby("File", as_index=False)
    .first()
    .rename(columns={
        "Payment date": "first_significant_payment_date",
        "Payment_amount": "first_significant_payment_amount"
    })
)

In [ ]:
first_payment_overall_df = sequence_client_df.merge(
    first_significant_payment_df[[
        "File",
        "first_significant_payment_date",
        "first_significant_payment_amount"
    ]],
    on="File",
    how="left"
)

In [ ]:
# Ja maksājums ir vienāds vai lielāks par 2.78% no Principal, tad tas tiek uzskatīts par nozīmīgu maksājumu, un tiek piešķirta vērtība 1, pretējā gadījumā 0
first_payment_overall_df["is_significant_payment"] = (
    first_payment_overall_df["first_significant_payment_date"].notna()
).astype(int)

first_payment_overall_df["days_to_first_significant_payment"] = (
    first_payment_overall_df["first_significant_payment_date"] - first_payment_overall_df["Listed"]
).dt.days

In [ ]:
# Ja bija maksājums, tad beigu datums ir maksājuma datums. Ja nebija, tad 3 gadi no Listed
first_payment_overall_df["sequence_end_date"] = np.where(
    first_payment_overall_df["is_significant_payment"] == 1,
    first_payment_overall_df["first_significant_payment_date"],
    first_payment_overall_df["observation_end"]
)

first_payment_overall_df["sequence_end_date"] = pd.to_datetime(
    first_payment_overall_df["sequence_end_date"]
)

In [ ]:
conditions = [
    first_payment_overall_df["days_to_first_significant_payment"].isna(),
    first_payment_overall_df["days_to_first_significant_payment"].between(0, 21, inclusive="both"),
    first_payment_overall_df["days_to_first_significant_payment"].between(22, 30, inclusive="both"),
    first_payment_overall_df["days_to_first_significant_payment"].between(31, 60, inclusive="both"),
    first_payment_overall_df["days_to_first_significant_payment"].between(61, 90, inclusive="both"),
    first_payment_overall_df["days_to_first_significant_payment"].between(91, 120, inclusive="both"),
    first_payment_overall_df["days_to_first_significant_payment"].between(121, 134, inclusive="both"),
    first_payment_overall_df["days_to_first_significant_payment"] >= 135
]

choices = [0, 1, 2, 3, 4, 5, 6, 7]

first_payment_overall_df["days_to_first_significant_payment_interval_id"] = np.select(
    conditions,
    choices,
    default=0
).astype(int)

In [ ]:
sequence_events_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile//2026//Valuation data\final_relation_clean_df.csv"
sequence_events_df = pd.read_csv(sequence_events_path, sep=';')
sequence_events_df = sequence_events_df[['File', 'Type', 'Contact_date']]

In [ ]:
# Inner join, jo mūs interesē tikai aktivitāšu secības (ir lietas bez aktivitātēm - nepamatoti cedētas)
clients_with_events = first_payment_overall_df.merge(
    sequence_events_df,
    on="File",
    how="inner"
)

In [ ]:
events_before_payment = clients_with_events[
    (clients_with_events["Contact_date"] >= clients_with_events["Listed"]) &
    (clients_with_events["Contact_date"] < clients_with_events["sequence_end_date"])
].copy()

In [ ]:
action_sequence_df = (
    events_before_payment
    .sort_values(["File", "Contact_date"])
    .groupby("File")["Type"]
    .apply(lambda x: " → ".join(x))
    .reset_index()
    .rename(columns={"Type": "action_sequence"})
)

In [ ]:
# Apvienojam maksājumus ar sagrupētām aktivitātēm
sequence_result_df = first_payment_overall_df.merge(
    action_sequence_df,
    on="File",
    how="left"
)

# sequence_result_df["action_sequence"] = (
#     sequence_result_df["action_sequence"]
#     .fillna("NO_ACTION")
# )

In [ ]:
# sequence_result_df["payment_result_event"] = np.select(
#     [
#         sequence_result_df["is_payment"] == 0,
#         sequence_result_df["is_payment_5pct_or_more"] == 1,
#         sequence_result_df["is_payment_5pct_or_more"] == 0
#     ],
#     [
#         "NO_PAYMENT",
#         "PAYMENT_5PCT_OR_MORE",
#         "PAYMENT_LESS_THAN_5PCT"
#     ],
#     default="UNKNOWN"
# )

In [ ]:

# Summary by outcome group and action sequence
# sequence_group_summary = (
#     sequence_result_df
#     .groupby(["payment_result_event", "action_sequence"])
#     .agg(
#         cases=("File", "count"),
#         avg_days_to_first_significant_payment=("days_to_first_significant_payment", "mean"),
#         median_days_to_first_significant_payment=("days_to_first_significant_payment", "median"),
#         avg_first_significant_payment_amount=("first_significant_payment_amount", "mean"),
#         median_first_significant_payment_amount=("first_significant_payment_amount", "median"),
#         avg_principal=("Principal", "mean"),
#         median_principal=("Principal", "median")
#     )
#     .reset_index()
#     .sort_values(["payment_result_event", "cases"], ascending=[True, False])
# )

# sequence_group_summary.head(30)

In [ ]:
sequence_summary = (
    sequence_result_df
    .groupby("action_sequence")
    .agg(
        cases=("File", "count"),
        payment_rate=("is_significant_payment", "mean"),
        avg_days_to_first_significant_payment=("days_to_first_significant_payment", "mean"),
        avg_first_significant_payment_amount=("first_significant_payment_amount", "mean"),
        avg_principal=("Principal", "mean")
    )
    .reset_index()
    .sort_values("cases", ascending=False)
)
